In [ ]:
from pathlib import Path
from xml.etree import ElementTree as ET

# import glasbey
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from networkx import DiGraph, Graph
from tqdm import tqdm

In [ ]:
base_path = Path(r"C:\FeedbackControl\pyclm\data\mdck\barspeeds")
track_path = base_path / "tracks"

In [ ]:
from collections import defaultdict

import matplotlib as mpl

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
fig2, ax2 = plt.subplots(1, 1, figsize=(8, 6))
# ax.set_ylim(0, 0.5)

experiments = defaultdict(list)

subset_dfs = []
barspeeds = []

pal = plt.get_cmap("viridis")

for src in tqdm(list(track_path.iterdir())):
    if not src.name.endswith(".csv"):
        continue

    spots_df = pd.read_csv(src)

    spots_df["id"] = spots_df.index
    spots_df["parent"] = spots_df.groupby("track_id").shift(1)["id"]
    spots_df["parent"] = spots_df["parent"].fillna(-1).astype(int)
    spots_df["um_x"] = spots_df["px_x"] * 1.33
    spots_df["um_y"] = spots_df["px_y"] * 1.33
    spots_df["t"] = spots_df["frame"] * 5
    spots_df["t_hour"] = spots_df["t"] // 60
    spots_df["t_2hour"] = spots_df["t"] // 120 * 2
    spots_df["dif_frame"] = (
        spots_df["frame"] - spots_df.groupby("track_id").shift(1)["frame"]
    )

    if src.stem[:3] == "dar":
        bar_speed = 0.0
        replicate = int(src.stem.split(".")[2][:2])
        barspeeds.append("dark")
    elif src.stem[:3] == "dow":
        bar_speed = 0.0
        replicate = int(src.stem.split(".")[2][:2])
        barspeeds.append("FBC")
    elif src.stem[:3] == "bar":
        barstr = src.stem.split(".")[0][3:]
        bar_speed = int(barstr) / (10 ** (len(barstr) - 1))

        replicate = int(src.stem.split(".")[2][:2])

        if bar_speed == 0:
            bar_speed = 0.00000001

        barspeeds.append(bar_speed)

        period = 100
        period_time = period / bar_speed
        duty_cycle = 0.2

        spots_df["phase"] = (
            (spots_df["t"] - (spots_df["um_y"] / bar_speed)) % period_time
        ) / period_time
        spots_df["phase_coarse"] = (spots_df["phase"] // 0.1) * 0.1
        spots_df["phase_fine"] = (spots_df["phase"] // 0.05) * 0.05

        spots_df["Relative Cell Position"] = [
            -(p - 0.1) if p < 0.6 else -(p - 0.1) + 1.0 for p in spots_df["phase"]
        ]
        # all_dfs["Relative Cell Positions"]
        spots_df["Relative Cell Position"] = pd.cut(
            spots_df["Relative Cell Position"], bins=15
        )
        spots_df["Relative Cell Position"] = [
            (p.right + p.left) / 2 for p in spots_df["Relative Cell Position"]
        ]
        spots_df["Relative Cell Position"] = (
            spots_df["Relative Cell Position"] * -2 * np.pi
        )

    else:
        continue

    spots_df["speed"] = bar_speed
    spots_df["replicate"] = replicate

    for col in ["px_x", "px_y", "um_x", "um_y"]:
        spots_df[f"parent_{col}"] = spots_df.groupby("track_id").shift(1)[col]
        spots_df[f"dif_{col}"] = (spots_df[col] - spots_df[f"parent_{col}"]) / spots_df[
            "dif_frame"
        ]
        spots_df[f"{col}_per_min"] = spots_df[f"dif_{col}"] / 5
        spots_df[f"{col}_per_h"] = spots_df[f"dif_{col}"] * 12

    spots_df["um_speed"] = np.sqrt(
        spots_df["um_x_per_min"] ** 2 + spots_df["um_y_per_min"] ** 2
    )

    spots_df = spots_df[spots_df["um_speed"] < 2]

    # spots_df["dif_px_x"] = spots_df["dif_px_x"] / spots_df["dif_frame"]

    subset = spots_df[
        (spots_df["px_x"] > 200)
        & (spots_df["px_x"] < 600)
        & (spots_df["px_y"] > 200)
        & (spots_df["px_y"] < 600)
        & (spots_df["t_hour"].between(7, 14))
        # (spots_df["um_speed"] > spots_df["um_speed"].quantile(0.90))
    ].copy()

    subset_dfs.append(subset)

    sns.lineplot(
        data=subset,
        x="t_hour",
        y="um_y_per_min",
        errorbar=None,
        color=pal(bar_speed / 2.0),
        label=f"{bar_speed * 60: 0.1f} um/h",
        estimator="mean",
        ax=ax,
        legend=False,
    )

    if bar_speed > 0:
        sns.lineplot(
            data=subset,
            x="phase_coarse",
            y="um_y_per_min",
            errorbar=None,
            color=pal(bar_speed / 2.0),
            label=f"{bar_speed * 60: 0.1f} um/h",
            estimator="mean",
            ax=ax2,
            legend=False,
        )

    experiments["bar_speed"].append(bar_speed)
    experiments["avg_speed"].append(
        subset[subset["t_hour"].between(3, 7)]["um_y_per_min"].mean()
    )
    experiments["median_speed"].append(
        subset[subset["t_hour"].between(3, 7)]["um_y_per_min"].median()
    )
    experiments["top_speed"].append(
        subset[subset["t_hour"].between(3, 7)]["um_y_per_min"].quantile(0.9)
    )
    experiments["n_cells"].append(
        subset[subset["t_hour"].between(3, 7)].groupby("frame")["um_x"].count().mean()
    )
    experiments["avg_speed_any"].append(
        subset[subset["t_hour"].between(3, 7)]["um_speed"].mean()
    )
    experiments["batch"].append(batch)
    experiments["replicate"].append(replicate)

cmap = plt.get_cmap("viridis")
norm = mpl.colors.Normalize(vmin=0, vmax=2.0)

# Create the ScalarMappable and add the colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("bar speed (um/min)")
ax.set_ylabel("Avg cell dy (um/min)")
ax.set_xlabel("Time (h)")

# ax = plt.gca()
# handles, labels = ax.get_legend_handles_labels()
# new_labels = [label[:3] for label in labels]
# ax.legend(handles, new_labels, title = 'Phase')

plt.show()

# sns.lineplot(data=subset, x="phase_coarse", y="um_y_per_min", )
plt.show()

# sns.histplot(data=spots_df, x="um_speed", bins=100)
# plt.show()
# spots_df

In [ ]:
merge_df = []

for subset_df, bar_speed in zip(subset_dfs, barspeeds, strict=False):
    if bar_speed in [0.25, 1.0, "dark", "FBC"]:
        subset_df["bar_speed"] = bar_speed
        if not isinstance(bar_speed, str):
            subset_df["bar_speed"] = np.round(bar_speed * 60).astype(int)

        print(subset_df["replicate"].unique())
        if subset_df["replicate"].unique()[0] > 3:
            continue

        merge_df.append(subset_df)


merge_df = pd.concat(merge_df, ignore_index=True)

merge_df["um_y_per_h"] = merge_df["um_y_per_min"] * 60

sns.violinplot(
    merge_df,
    x="bar_speed",
    y="um_y_per_h",
    hue="replicate",
    bw_adjust=0.02,
    order=[15, 60, "dark", "FBC"],
    legend=False,
)
plt.ylim(-50, 75)
plt.ylabel("Cell vertical speed (um/h)")
plt.xlabel("Bar speed (um/h)")
plt.axhline(0, color="k")
plt.show()